# 02 · Capa Gold: grilla hexagonal H3 y análisis exploratorio espacial (AEDE)

**Área de estudio:** corredor minero La Pampa, Huepetuhe y Mazuko, al sur de Madre de Dios. El rectángulo de
análisis incluye sectores limítrofes de Cusco y Puno; cada hexágono lleva su departamento, provincia y distrito, y
los mapas muestran el límite departamental para no confundir territorios.

Millones de píxeles se agregan a una **grilla hexagonal discreta global H3**, nativa en Databricks SQL:

- Todos los hexágonos tienen la misma área y **6 vecinos equidistantes** (no hay ambigüedad torre/reina).
- `h3_longlatash3string` asigna cada píxel a su hexágono en paralelo, sin geometrías.
- Resolución 9 ≈ 0.105 km² (≈ 260 píxeles de 20 m por hexágono).

**Máscara de bosque.** Las métricas de perturbación y pérdida se calculan solo sobre píxeles que eran
**bosque en el año base** según Geobosques. Así los cauces, playas de río, poblados y áreas sin bosque no se
confunden con minería, y las tasas se expresan **sobre el bosque disponible**, no sobre el área total.

Al final se exporta el **dataset limpio** (Parquet, CSV y GeoJSON) que se usa en la demostración.

In [ ]:
%pip install -q rasterio pyproj h3 geopandas shapely esda libpysal spreg pykrige scikit-gstat mapclassify
dbutils.library.restartPython()

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

In [ ]:
from pyspark.sql import functions as F
silver = spark.table(dbx.tbl("silver_s2_pixeles"))
hansen = spark.table(dbx.tbl("silver_hansen_pixeles"))
YEARS = sorted(r.year for r in silver.select("year").distinct().collect())
Y0, Y1 = YEARS[0], YEARS[-1]
R = C.H3_RES_FINE
HAS_GB = spark.catalog.tableExists(dbx.tbl("silver_geobosques_pixeles"))
silver.createOrReplaceTempView("px")
hansen.createOrReplaceTempView("hx")
if HAS_GB:
    spark.table(dbx.tbl("silver_geobosques_pixeles")).createOrReplaceTempView("gbx")
print("Años:", YEARS, "| Geobosques:", HAS_GB)

## 1. Agregación píxel → hexágono (Spark SQL + H3 nativo)

**Bosque en el año base** (`bosque_y0`): cobertura Geobosques 2, 3 o 4 (bosque en 2000) y sin pérdida registrada
hasta el año base. **Píxel perturbado:** NDVI < 0.3 con suelo desnudo (BSI > 0) o agua turbia (MNDWI > 0),
la huella de playas de lavado y pozas de relave. `frac_pert` = fracción del bosque del año base que aparece
perturbada en cada año.

In [ ]:
y0_code = Y0 - 2000
if HAS_GB:
    px_src = f"""SELECT p.*, COALESCE(g.cobertura_gb IN (2, 3, 4) AND NOT (g.perdida_gb BETWEEN 1 AND {y0_code}), TRUE) AS bosque_y0
                 FROM px p LEFT JOIN gbx g ON p.row = g.row AND p.col = g.col"""
else:
    px_src = "SELECT p.*, TRUE AS bosque_y0 FROM px p"

gold_anual = spark.sql(f"""
SELECT h3_longlatash3string(lon, lat, {R}) AS h3, year,
       COUNT(*)                                        AS n_px,
       SUM(CASE WHEN bosque_y0 THEN 1 ELSE 0 END)      AS n_px_bosque,
       AVG(n_obs) AS n_obs, AVG(ndvi) AS ndvi, AVG(nbr) AS nbr, AVG(mndwi) AS mndwi, AVG(bsi) AS bsi,
       AVG(CASE WHEN bosque_y0 THEN (CASE WHEN ndvi < 0.3 AND (bsi > 0 OR mndwi > 0) THEN 1D ELSE 0D END) END) AS frac_pert,
       AVG(CASE WHEN mndwi > 0 THEN 1D ELSE 0D END)    AS frac_agua
FROM ({px_src}) t
GROUP BY 1, 2
""")
dbx.save_table(gold_anual, "gold_h3_anual")

gold_hansen = spark.sql(f"""
SELECT h3_longlatash3string(lon, lat, {R}) AS h3,
       COUNT(*)           AS n_px_total,
       AVG(treecover2000) AS treecover2000,
       AVG(CASE WHEN lossyear BETWEEN {y0_code + 1} AND {min(Y1, 2024) - 2000} THEN 1D ELSE 0D END) AS frac_perdida_hansen,
       SUM(CASE WHEN treecover2000 >= 30 AND lossyear BETWEEN {y0_code + 1} AND {min(Y1, 2024) - 2000} THEN 1 ELSE 0 END)
         / NULLIF(SUM(CASE WHEN treecover2000 >= 30 AND NOT (lossyear BETWEEN 1 AND {y0_code}) THEN 1 ELSE 0 END), 0)
                          AS tasa_perdida_hansen
FROM hx GROUP BY 1
HAVING n_px_total >= 150
""")
dbx.save_table(gold_hansen, "gold_h3_hansen")

if HAS_GB:
    codes = ",".join(map(str, C.geobosques_codes(Y0, Y1)))
    gold_gb = spark.sql(f"""
    SELECT h3_longlatash3string(lon, lat, {R}) AS h3,
           AVG(CASE WHEN perdida_gb IN ({codes}) THEN 1D ELSE 0D END)  AS frac_perdida_oficial,
           SUM(CASE WHEN perdida_gb IN ({codes}) THEN 1 ELSE 0 END)
             / NULLIF(SUM(CASE WHEN cobertura_gb IN (2, 3, 4) AND NOT (perdida_gb BETWEEN 1 AND {y0_code}) THEN 1 ELSE 0 END), 0)
                                                                        AS tasa_perdida_oficial,
           AVG(CASE WHEN perdida_gb = {C.GEOBOSQUES_NATURAL_2025} THEN 1D ELSE 0D END) AS frac_perdida_natural_2025,
           AVG(CASE WHEN cobertura_gb IN (2, 3, 4) AND NOT (perdida_gb BETWEEN 1 AND {y0_code}) THEN 1D ELSE 0D END)
                                                                        AS frac_bosque_y0,
           AVG(CASE WHEN cobertura_gb = 4 THEN 1D ELSE 0D END)         AS frac_bosque_2025,
           AVG(CASE WHEN cobertura_gb = 1 THEN 1D ELSE 0D END)         AS frac_hidrografia,
           AVG(CASE WHEN riesgo_gb BETWEEN 1 AND 5 THEN riesgo_gb END) AS riesgo_deforestacion
    FROM gbx GROUP BY 1""")
    dbx.save_table(gold_gb, "gold_h3_geobosques")

## 2. Tabla analítica ancha: un hexágono por fila

- NDVI de un año: faltante si menos del 50 % de los píxeles del hexágono tuvo una observación sin nube.
- Perturbación de un año: faltante si el hexágono tiene menos de 30 píxeles de bosque en el año base.

In [ ]:
ga = spark.table(dbx.tbl("gold_h3_anual"))
gh = spark.table(dbx.tbl("gold_h3_hansen"))
valid = (ga.join(gh.select("h3", "n_px_total"), "h3")
         .withColumn("ndvi", F.when(F.col("n_px") / F.col("n_px_total") >= 0.5, F.col("ndvi")))
         .withColumn("frac_pert", F.when(F.col("n_px_bosque") >= 30, F.col("frac_pert"))))
wide = (valid.groupBy("h3").pivot("year", YEARS)
        .agg(F.first("ndvi").alias("ndvi"), F.first("frac_pert").alias("pert"), F.first("n_obs").alias("nobs")))
for y in YEARS:
    for m in ["ndvi", "pert", "nobs"]:
        wide = wide.withColumnRenamed(f"{y}_{m}", f"{m}_{y}")
base_h3 = gh.join(spark.table(dbx.tbl("gold_h3_geobosques")), "h3", "left") if HAS_GB else gh
wide = (base_h3.join(wide, "h3", "left")
        .withColumn("d_ndvi", F.col(f"ndvi_{Y1}") - F.col(f"ndvi_{Y0}"))
        .withColumn("d_pert", F.col(f"pert_{Y1}") - F.col(f"pert_{Y0}")))
feat = wide.toPandas().pipe(dbx.clean)
print(f"Hexágonos: {len(feat):,} | con NDVI {Y1}: {feat[f'ndvi_{Y1}'].notna().sum():,} "
      f"| con perturbación medible: {feat['d_pert'].notna().sum():,}")

## 3. Covariables geográficas

Hipótesis de trabajo sobre la minería aluvial:
- se concentra cerca de la **red de drenaje** (ríos y quebradas: el oro está en los sedimentos),
- se expande desde la **Carretera Interoceánica** (acceso),
- es menor dentro de **áreas protegidas** y presiona sobre su **zona de amortiguamiento**.

In [ ]:
import h3, numpy as np, geopandas as gpd

lat_lon = np.array([h3.cell_to_latlng(c) for c in feat.h3])
feat["lat"], feat["lon"] = lat_lon[:, 0], lat_lon[:, 1]
pts = gpd.GeoDataFrame(feat, geometry=gpd.points_from_xy(feat.lon, feat.lat), crs="EPSG:4326").to_crs(C.CRS_UTM)
feat["x_km"], feat["y_km"] = pts.geometry.x / 1000, pts.geometry.y / 1000

vec = Path(VOL) / "bronze" / "vector"
def load(name):
    p = vec / f"{name}.geojson"
    return gpd.read_file(p).to_crs(C.CRS_UTM) if p.exists() else None

vias, rios, cauces = load("vias"), load("rios"), load("cauces")
anp, za = load("areas_protegidas_sernanp"), load("zonas_amortiguamiento")
dist = load("distritos")
if dist is None:
    dist = load("distritos_madre_de_dios")
if anp is None:
    anp = load("areas_protegidas")
if vias is not None:
    principales = vias[vias["highway"].isin(["trunk", "primary"])] if "highway" in vias else vias
    feat["dist_via_km"] = pts.distance(principales.union_all()).values / 1000
if rios is not None:
    feat["dist_rio_km"] = pts.distance(rios.union_all()).values / 1000
if cauces is not None:
    feat["dist_cauce_km"] = pts.distance(cauces.union_all()).values / 1000
if anp is not None and len(anp):
    feat["en_anp"] = pts.within(anp.union_all()).astype(int).values
    feat["dist_anp_km"] = pts.distance(anp.union_all()).values / 1000
if za is not None and len(za):
    feat["en_za"] = pts.within(za.union_all()).astype(int).values
    feat["dist_za_km"] = pts.distance(za.union_all()).values / 1000
if dist is not None:
    cols_adm = [c for c in ["departamento", "provincia", "distrito"] if c in dist]
    adm = gpd.sjoin(pts[["geometry"]], dist[cols_adm + ["geometry"]], how="left", predicate="within")
    adm = adm[cols_adm].groupby(level=0).first()
    for c in cols_adm:
        feat[c] = adm[c].values
    if "departamento" in feat:
        rep = feat.groupby("departamento", dropna=False).agg(hexagonos=("h3", "size"),
                                                            distritos=("distrito", lambda x: ", ".join(sorted(x.dropna().unique()))))
        rep["% del área"] = (100 * rep.hexagonos / len(feat)).round(1)
        print("Cobertura del área de estudio por departamento:")
        print(rep.to_string())
dbx.save_table(spark.createDataFrame(feat), "gold_h3_features")
feat.describe().T.round(3)

## 4. Exportación del dataset limpio para la demostración

In [ ]:
from geocloud import spatial
import shutil, tempfile
out = Path(VOL) / "gold"; out.mkdir(parents=True, exist_ok=True)
feat.to_parquet(out / "h3_features.parquet", index=False)
feat.to_csv(out / "h3_features.csv", index=False, encoding="utf-8")
tmp = Path(tempfile.mkdtemp()) / "h3_features.geojson"      # GDAL escribe en disco local y luego se copia al Volume
spatial.h3_to_geodataframe(feat).to_file(tmp, driver="GeoJSON")
shutil.copyfile(tmp, out / "h3_features.geojson")
print(sorted(p.name for p in out.iterdir()))

## 5. AEDE: distribución y validación cruzada de fuentes

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from geocloud import spatial

feat = spark.table(dbx.tbl("gold_h3_features")).toPandas().pipe(dbx.clean)
YEARS = sorted(int(c.split("_")[1]) for c in feat.columns if c.startswith("ndvi_") and c.split("_")[1].isdigit())
Y0, Y1 = YEARS[0], YEARS[-1]
gdf = spatial.h3_to_geodataframe(feat, crs_out=C.CRS_UTM)
# Hexágonos de bosque: al menos la mitad de su superficie era bosque en el año base (excluye ríos, poblados, no bosque)
BOSQUE = gdf["frac_bosque_y0"] >= 0.5 if "frac_bosque_y0" in gdf else pd.Series(True, index=gdf.index)
print(f"Hexágonos: {len(gdf):,} | de bosque en {Y0}: {BOSQUE.sum():,} | años: {YEARS}")

In [ ]:
# Capas de contexto cartográfico (leyenda, topónimos, escala y norte)
vec = Path(VOL) / "bronze" / "vector"
def capa(n):
    p = vec / f"{n}.geojson"
    return gpd.read_file(p).to_crs(C.CRS_UTM) if p.exists() else None

rios, vias, za, poblados = capa("rios"), capa("vias"), capa("zonas_amortiguamiento"), capa("poblados")
deptos = capa("departamentos")
anp = capa("areas_protegidas_sernanp")
anp = anp if anp is not None else capa("areas_protegidas")
if vias is not None and "highway" in vias:
    vias = vias[vias["highway"].isin(["trunk", "primary"])]
if poblados is not None and len(poblados) > 15:
    poblados = poblados[poblados["fclass"].isin(["city", "town"])] if "fclass" in poblados else poblados.head(12)
BOUNDS = C.GRID_BOUNDS_UTM
CONTEXTO = {"Límite departamental": (deptos, dict(color="#4d4d4d", lw=2.2, ls="-.")),
            "Ríos": (rios, dict(color="#1f78b4", lw=0.8)),
            "Vía principal (Interoceánica)": (vias, dict(color="black", lw=1.1)),
            "ANP (SERNANP)": (anp, dict(color="purple", lw=1.1)),
            "Zona de amortiguamiento": (za, dict(color="darkorange", lw=1.0, ls="--"))}

def contexto(ax, legend=True, **kw):
    spatial.decorate_map(ax, BOUNDS, CONTEXTO, poblados, legend=legend, **kw)
    if deptos is not None:
        from shapely.geometry import box
        caja = box(*BOUNDS)
        for _, r in deptos.iterrows():
            parte = r.geometry.intersection(caja)
            if parte.is_empty or parte.area < 0.03 * caja.area:
                continue
            p = parte.representative_point()
            ax.text(p.x, p.y, str(r["departamento"]).upper(), fontsize=11, color="#4d4d4d", alpha=0.75,
                    ha="center", va="center", fontweight="bold", style="italic", zorder=8)

In [ ]:
from scipy.stats import spearmanr
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for y in YEARS:
    ax[0].hist(gdf[f"ndvi_{y}"].dropna(), bins=60, range=(-0.2, 1), alpha=0.45, label=str(y), density=True)
ax[0].set_title("NDVI por hexágono y año"); ax[0].set_xlabel("NDVI"); ax[0].legend()
ax[1].hist(gdf.loc[BOSQUE, "d_ndvi"].dropna(), bins=80, range=(-1, 0.6), color="#8c2d04")
ax[1].axvline(0, c="k", lw=0.8); ax[1].set_title(f"Cambio de NDVI {Y0}-{Y1} (hexágonos de bosque)")
REF = "tasa_perdida_oficial" if "tasa_perdida_oficial" in gdf else "tasa_perdida_hansen"
v = gdf.loc[BOSQUE, [REF, "d_pert"]].dropna()
rho = spearmanr(v[REF], v["d_pert"]).correlation
hb = ax[2].hexbin(v[REF], v["d_pert"], gridsize=40, bins="log", cmap="viridis", mincnt=1)
plt.colorbar(hb, ax=ax[2], label="hexágonos (log)")
ax[2].set_xlabel(f"{REF} (fracción del bosque {Y0})"); ax[2].set_ylabel("Δ fracción perturbada (Sentinel-2)")
ax[2].set_title(f"Validación Sentinel-2 vs referencia oficial, Spearman ρ = {rho:.2f}")
plt.tight_layout(); plt.savefig(FIG / "02_distribuciones.png", dpi=150); plt.show()

**Lectura de la validación.** Sentinel-2 detecta la perturbación tipo minería (suelo desnudo y pozas), mientras
que la pérdida oficial incluye además la conversión a agricultura y pastos, que mantiene vegetación. Por eso
hay hexágonos con pérdida oficial alta y Δ perturbación cercana a cero: no es un error, son dos fenómenos distintos.

In [ ]:
def mapa(ax, col, cmap, title, vmin=None, vmax=None, mask=None, legend=False):
    g = gdf if mask is None else gdf[mask]
    gdf.plot(ax=ax, color="#e6e6e6", linewidth=0)
    g.plot(column=col, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, linewidth=0, legend=True,
           missing_kwds={"color": "#e6e6e6"}, legend_kwds={"shrink": 0.6})
    contexto(ax, legend=legend)
    ax.set_title(title)

fig, ax = plt.subplots(2, 2, figsize=(17, 11))
mapa(ax[0, 0], f"ndvi_{Y0}", "RdYlGn", f"NDVI {Y0}", 0, 0.9, legend=True)
mapa(ax[0, 1], f"ndvi_{Y1}", "RdYlGn", f"NDVI {Y1}", 0, 0.9)
mapa(ax[1, 0], "d_ndvi", "RdBu", f"Cambio de NDVI {Y0}-{Y1}", -0.6, 0.6, mask=BOSQUE)
if "tasa_perdida_oficial" in gdf:
    mapa(ax[1, 1], "tasa_perdida_oficial", "Reds", f"Pérdida antrópica oficial {Y0+1}-{Y1} (Geobosques)", 0, 1, mask=BOSQUE)
else:
    mapa(ax[1, 1], "tasa_perdida_hansen", "Reds", f"Pérdida de bosque {Y0+1}-{min(Y1, 2024)} (Hansen)", 0, 1, mask=BOSQUE)
plt.tight_layout(); plt.savefig(FIG / "02_mapas.png", dpi=150); plt.show()

## 6. Gradientes: drenaje y acceso

Las tasas se calculan sobre el bosque disponible en el año base (hexágonos de bosque), para no penalizar las zonas
pegadas a los ríos, donde parte del área es agua o playa.

In [ ]:
bins = [0, 0.5, 1, 2, 3, 5, 8, 12, np.inf]
cols = {"d_pert": "Δ perturbación Sentinel-2 (minería)",
        "tasa_perdida_oficial": "Pérdida antrópica oficial", "tasa_perdida_hansen": "Pérdida Hansen"}
cols = {k: v for k, v in cols.items() if k in gdf}
dist_cols = [c for c in [("dist_cauce_km", "Distancia al cauce más cercano: ríos y quebradas (km)"),
                         ("dist_via_km", "Distancia a la vía principal (km)")] if c[0] in gdf]
fig, axes = plt.subplots(1, len(dist_cols), figsize=(8 * len(dist_cols), 4.5), squeeze=False)
tablas = {}
for ax, (dc, lab) in zip(axes[0], dist_cols):
    g = gdf[BOSQUE].copy()
    g["banda"] = pd.cut(g[dc], bins)
    t = g.groupby("banda", observed=True)[list(cols)].mean().rename(columns=cols)
    t.plot.bar(ax=ax, width=0.8, color=["#d95f02", "#1b9e77", "#7570b3"][:len(cols)])
    ax.set_xlabel(lab); ax.set_ylabel("Fracción del bosque del año base"); ax.tick_params(axis="x", rotation=0)
    ax.legend(fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.2), frameon=False)
    tablas[dc] = t.assign(hexagonos=g.groupby("banda", observed=True).size())
axes[0, 0].set_title("Gradiente según la red de drenaje")
if len(dist_cols) > 1:
    axes[0, 1].set_title("Gradiente según el acceso vial")
plt.tight_layout(); plt.savefig(FIG / "02_gradientes.png", dpi=150, bbox_inches="tight"); plt.show()
for k, t in tablas.items():
    print(k); print(t.round(3), "\n")

### Limitaciones observadas

- **NDVI 2024:** franja diagonal más clara, el borde de pasada entre órbitas (efecto de geometría de iluminación).
  No afecta la comparación principal, que es entre el año base y el último año.
- **Geobosques:** en el norte se ven rayas verticales propias del producto Landsat de origen; Sentinel-2 (10-20 m)
  ofrece bordes más nítidos.
- La perturbación por índices espectrales no distingue minería de otros suelos desnudos (cantera, vía nueva); por
  eso se valida con la fuente oficial y se interpreta junto con las covariables.